# CSV Ingestion Demo — Superstore Sales Dataset

## Flow

```text
CSV file
↓
Load in Python
↓
Display first 5 rows
↓
Clean column names
↓
Validate required fields
↓
Separate required-field missing values and optional-field missing values
↓
Remove duplicate rows
↓
Save raw output
↓
Save staging output
↓
Save clean output
↓
Generate ingestion log
```

## Expected Input

```text
data/sample_inputs/Superstore.csv
```

## Expected Outputs

```text
data/raw/csv/superstore_raw.csv
data/staging/csv/superstore_staging.csv
data/clean/csv/superstore_clean.csv
logs/csv_ingestion_log.json
```

## 1. Import Required Libraries

In [1]:
import pandas as pd
import json
import uuid
import re
import shutil

from pathlib import Path
from datetime import datetime, timezone

## 2. Define Project Paths

In [2]:
def find_project_root(current_path: Path) -> Path:
    current_path = current_path.resolve()

    for path in [current_path] + list(current_path.parents):
        if (path / "data").exists():
            return path

    raise FileNotFoundError("Project root not found. Make sure data/ folder exists.")


def to_relative_path(path: Path, project_root: Path) -> str:
    return path.resolve().relative_to(project_root.resolve()).as_posix()


CURRENT_DIR = Path.cwd()
PROJECT_ROOT = find_project_root(CURRENT_DIR)

input_path = PROJECT_ROOT / "data" / "sample_inputs" / "Superstore.csv"

raw_dir = PROJECT_ROOT / "data" / "raw" / "csv"
staging_dir = PROJECT_ROOT / "data" / "staging" / "csv"
clean_dir = PROJECT_ROOT / "data" / "clean" / "csv"
log_dir = PROJECT_ROOT / "logs"

raw_dir.mkdir(parents=True, exist_ok=True)
staging_dir.mkdir(parents=True, exist_ok=True)
clean_dir.mkdir(parents=True, exist_ok=True)
log_dir.mkdir(parents=True, exist_ok=True)

raw_output_path = raw_dir / "superstore_raw.csv"
staging_output_path = staging_dir / "superstore_staging.csv"
clean_output_path = clean_dir / "superstore_clean.csv"
log_output_path = log_dir / "csv_ingestion_log.json"

print("Current dir:", CURRENT_DIR)
print("Project root:", PROJECT_ROOT)
print("Input path:", input_path)
print("Input exists:", input_path.exists())
print("Raw output:", raw_output_path)
print("Staging output:", staging_output_path)
print("Clean output:", clean_output_path)
print("Log output:", log_output_path)

Current dir: f:\data\new\quanskill\DataVision_Duy\week2\notebooks\data_team
Project root: F:\data\new\quanskill\DataVision_Duy\week2
Input path: F:\data\new\quanskill\DataVision_Duy\week2\data\sample_inputs\Superstore.csv
Input exists: True
Raw output: F:\data\new\quanskill\DataVision_Duy\week2\data\raw\csv\superstore_raw.csv
Staging output: F:\data\new\quanskill\DataVision_Duy\week2\data\staging\csv\superstore_staging.csv
Clean output: F:\data\new\quanskill\DataVision_Duy\week2\data\clean\csv\superstore_clean.csv
Log output: F:\data\new\quanskill\DataVision_Duy\week2\logs\csv_ingestion_log.json


## 3. Validate Input File

In [3]:
if not input_path.exists():
    raise FileNotFoundError(f"File not found: {input_path}")

if input_path.stat().st_size == 0:
    raise ValueError("CSV file is empty.")

print("Input validation passed.")

Input validation passed.


## 4. Helper Functions

In [4]:
def clean_column_name(column_name: str) -> str:
    column_name = str(column_name).strip().lower()
    column_name = re.sub(r"[^a-z0-9]+", "_", column_name)
    column_name = re.sub(r"_+", "_", column_name)
    return column_name.strip("_")


def read_csv_with_fallback_encoding(file_path: Path) -> pd.DataFrame:
    encodings = ["utf-8", "utf-8-sig", "latin1", "cp1252"]

    last_error = None

    for encoding in encodings:
        try:
            print(f"Trying encoding: {encoding}")
            df = pd.read_csv(file_path, encoding=encoding)
            print(f"Success with encoding: {encoding}")
            return df
        except UnicodeDecodeError as error:
            last_error = error
            print(f"Failed encoding: {encoding}")

    raise ValueError(f"Cannot read CSV. Last error: {last_error}")

## 5. Start Ingestion Run

In [5]:
run_id = str(uuid.uuid4())
source_name = "superstore_sales_csv"
source_type = "csv"
owner = "Nguyen Minh Duy"

start_time = datetime.now(timezone.utc).isoformat()

print("Run ID:", run_id)
print("Start time:", start_time)

Run ID: a21ea7fa-6734-477b-b3d1-e0403f4f23ce
Start time: 2026-06-14T04:51:29.519871+00:00


## 6. Load Sample CSV

In [6]:
try:
    df_raw = read_csv_with_fallback_encoding(input_path)
    status = "success"
    error_message = None
except Exception as error:
    status = "failed"
    error_message = str(error)
    raise

records_read = len(df_raw)

print("CSV loaded successfully.")
print("Records read:", records_read)
print("Column count:", len(df_raw.columns))
print("Columns:", df_raw.columns.tolist())

Trying encoding: utf-8
Failed encoding: utf-8
Trying encoding: utf-8-sig
Failed encoding: utf-8-sig
Trying encoding: latin1
Success with encoding: latin1
CSV loaded successfully.
Records read: 9994
Column count: 21
Columns: ['Row ID', 'Order ID', 'Order Date', 'Ship Date', 'Ship Mode', 'Customer ID', 'Customer Name', 'Segment', 'Country', 'City', 'State', 'Postal Code', 'Region', 'Product ID', 'Category', 'Sub-Category', 'Product Name', 'Sales', 'Quantity', 'Discount', 'Profit']


## 7. Display First 5 Rows

In [7]:
df_raw.head(5)

,Row ID,Order ID,Order Date,Ship Date,Ship Mode,Customer ID,Customer Name,Segment,Country,City,...,Postal Code,Region,Product ID,Category,Sub-Category,Product Name,Sales,Quantity,Discount,Profit
0,1,CA-2013-152156,09-11-2013,12-11-2013,Second Class,CG-12520,Claire Gute,Consumer,United States,Henderson,...,42420,South,FUR-BO-10001798,Furniture,Bookcases,Bush Somerset Collection Bookcase,261.9600,2,0.00,41.9136
1,2,CA-2013-152156,09-11-2013,12-11-2013,Second Class,CG-12520,Claire Gute,Consumer,United States,Henderson,...,42420,South,FUR-CH-10000454,Furniture,Chairs,"Hon Deluxe Fabric Upholstered Stacking Chairs,...",731.9400,3,0.00,219.5820
2,3,CA-2013-138688,13-06-2013,17-06-2013,Second Class,DV-13045,Darrin Van Huff,Corporate,United States,Los Angeles,...,90036,West,OFF-LA-10000240,Office Supplies,Labels,Self-Adhesive Address Labels for Typewriters b...,14.6200,2,0.00,6.8714
3,4,US-2012-108966,11-10-2012,18-10-2012,Standard Class,SO-20335,Sean O'Donnell,Consumer,United States,Fort Lauderdale,...,33311,South,FUR-TA-10000577,Furniture,Tables,Bretford CR4500 Series Slim Rectangular Table,957.5775,5,0.45,-383.0310
4,5,US-2012-108966,11-10-2012,18-10-2012,Standard Class,SO-20335,Sean O'Donnell,Consumer,United States,Fort Lauderdale,...,33311,South,OFF-ST-10000760,Office Supplies,Storage,Eldon Fold 'N Roll Cart System,22.3680,2,0.20,2.5164


## 8. Save Raw Copy

In [8]:
shutil.copy2(input_path, raw_output_path)

print("Raw saved:", to_relative_path(raw_output_path, PROJECT_ROOT))

Raw saved: data/raw/csv/superstore_raw.csv


## 9. Clean Column Names

In [9]:
df_staging = df_raw.copy()

original_columns = df_staging.columns.tolist()
cleaned_columns = [clean_column_name(col) for col in original_columns]

df_staging.columns = cleaned_columns

print("Original columns:")
print(original_columns)

print("\nCleaned columns:")
print(cleaned_columns)

df_staging.head(5)

Original columns:
['Row ID', 'Order ID', 'Order Date', 'Ship Date', 'Ship Mode', 'Customer ID', 'Customer Name', 'Segment', 'Country', 'City', 'State', 'Postal Code', 'Region', 'Product ID', 'Category', 'Sub-Category', 'Product Name', 'Sales', 'Quantity', 'Discount', 'Profit']

Cleaned columns:
['row_id', 'order_id', 'order_date', 'ship_date', 'ship_mode', 'customer_id', 'customer_name', 'segment', 'country', 'city', 'state', 'postal_code', 'region', 'product_id', 'category', 'sub_category', 'product_name', 'sales', 'quantity', 'discount', 'profit']


,row_id,order_id,order_date,ship_date,ship_mode,customer_id,customer_name,segment,country,city,...,postal_code,region,product_id,category,sub_category,product_name,sales,quantity,discount,profit
0,1,CA-2013-152156,09-11-2013,12-11-2013,Second Class,CG-12520,Claire Gute,Consumer,United States,Henderson,...,42420,South,FUR-BO-10001798,Furniture,Bookcases,Bush Somerset Collection Bookcase,261.9600,2,0.00,41.9136
1,2,CA-2013-152156,09-11-2013,12-11-2013,Second Class,CG-12520,Claire Gute,Consumer,United States,Henderson,...,42420,South,FUR-CH-10000454,Furniture,Chairs,"Hon Deluxe Fabric Upholstered Stacking Chairs,...",731.9400,3,0.00,219.5820
2,3,CA-2013-138688,13-06-2013,17-06-2013,Second Class,DV-13045,Darrin Van Huff,Corporate,United States,Los Angeles,...,90036,West,OFF-LA-10000240,Office Supplies,Labels,Self-Adhesive Address Labels for Typewriters b...,14.6200,2,0.00,6.8714
3,4,US-2012-108966,11-10-2012,18-10-2012,Standard Class,SO-20335,Sean O'Donnell,Consumer,United States,Fort Lauderdale,...,33311,South,FUR-TA-10000577,Furniture,Tables,Bretford CR4500 Series Slim Rectangular Table,957.5775,5,0.45,-383.0310
4,5,US-2012-108966,11-10-2012,18-10-2012,Standard Class,SO-20335,Sean O'Donnell,Consumer,United States,Fort Lauderdale,...,33311,South,OFF-ST-10000760,Office Supplies,Storage,Eldon Fold 'N Roll Cart System,22.3680,2,0.20,2.5164


## 10. Define Required and Optional Fields

In [10]:
required_fields = [
    "row_id",
    "order_id",
    "order_date",
    "ship_date",
    "customer_id",
    "customer_name",
    "country",
    "city",
    "state",
    "region",
    "product_id",
    "category",
    "sub_category",
    "product_name",
    "sales",
    "quantity",
    "discount",
    "profit",
]

optional_fields = [
    "ship_mode",
    "segment",
    "postal_code",
]

print("Required fields:", required_fields)
print("Optional fields:", optional_fields)

Required fields: ['row_id', 'order_id', 'order_date', 'ship_date', 'customer_id', 'customer_name', 'country', 'city', 'state', 'region', 'product_id', 'category', 'sub_category', 'product_name', 'sales', 'quantity', 'discount', 'profit']
Optional fields: ['ship_mode', 'segment', 'postal_code']


## 11. Validate Required Columns

In [11]:
missing_required_columns = [
    col for col in required_fields
    if col not in df_staging.columns
]

if missing_required_columns:
    raise ValueError(f"Missing required columns: {missing_required_columns}")

print("Required column validation passed.")

Required column validation passed.


## 12. Check Missing Values

In [12]:
missing_values_all = df_staging.isna().sum()
required_missing_values = df_staging[required_fields].isna().sum()
optional_missing_values = df_staging[optional_fields].isna().sum()

total_missing_values = int(missing_values_all.sum())

print("All missing values:")
print(missing_values_all)

print("\nRequired missing values:")
print(required_missing_values)

print("\nOptional missing values:")
print(optional_missing_values)

print("\nTotal missing values:", total_missing_values)

All missing values:
row_id           0
order_id         0
order_date       0
ship_date        0
ship_mode        0
customer_id      0
customer_name    0
segment          0
country          0
city             0
state            0
postal_code      0
region           0
product_id       0
category         0
sub_category     0
product_name     0
sales            0
quantity         0
discount         0
profit           0
dtype: int64

Required missing values:
row_id           0
order_id         0
order_date       0
ship_date        0
customer_id      0
customer_name    0
country          0
city             0
state            0
region           0
product_id       0
category         0
sub_category     0
product_name     0
sales            0
quantity         0
discount         0
profit           0
dtype: int64

Optional missing values:
ship_mode      0
segment        0
postal_code    0
dtype: int64

Total missing values: 0


## 13. Save Parsed Output to Staging

In [13]:
df_staging.to_csv(staging_output_path, index=False, encoding="utf-8")

print("Staging saved:", to_relative_path(staging_output_path, PROJECT_ROOT))

Staging saved: data/staging/csv/superstore_staging.csv


## 14. Check Duplicate Rows

In [14]:
duplicate_count = int(df_staging.duplicated().sum())

print("Duplicate rows:", duplicate_count)

Duplicate rows: 0


## 15. Create Clean Data

In [15]:
df_clean = df_staging.copy()

# Clean data means validated data:
# - required fields must not be missing
# - duplicate rows are removed
df_clean = df_clean.dropna(subset=required_fields)
df_clean = df_clean.drop_duplicates()

records_valid = len(df_clean)
records_invalid = records_read - records_valid

print("Records read:", records_read)
print("Records valid:", records_valid)
print("Records invalid:", records_invalid)

df_clean.head(5)

Records read: 9994
Records valid: 9994
Records invalid: 0


,row_id,order_id,order_date,ship_date,ship_mode,customer_id,customer_name,segment,country,city,...,postal_code,region,product_id,category,sub_category,product_name,sales,quantity,discount,profit
0,1,CA-2013-152156,09-11-2013,12-11-2013,Second Class,CG-12520,Claire Gute,Consumer,United States,Henderson,...,42420,South,FUR-BO-10001798,Furniture,Bookcases,Bush Somerset Collection Bookcase,261.9600,2,0.00,41.9136
1,2,CA-2013-152156,09-11-2013,12-11-2013,Second Class,CG-12520,Claire Gute,Consumer,United States,Henderson,...,42420,South,FUR-CH-10000454,Furniture,Chairs,"Hon Deluxe Fabric Upholstered Stacking Chairs,...",731.9400,3,0.00,219.5820
2,3,CA-2013-138688,13-06-2013,17-06-2013,Second Class,DV-13045,Darrin Van Huff,Corporate,United States,Los Angeles,...,90036,West,OFF-LA-10000240,Office Supplies,Labels,Self-Adhesive Address Labels for Typewriters b...,14.6200,2,0.00,6.8714
3,4,US-2012-108966,11-10-2012,18-10-2012,Standard Class,SO-20335,Sean O'Donnell,Consumer,United States,Fort Lauderdale,...,33311,South,FUR-TA-10000577,Furniture,Tables,Bretford CR4500 Series Slim Rectangular Table,957.5775,5,0.45,-383.0310
4,5,US-2012-108966,11-10-2012,18-10-2012,Standard Class,SO-20335,Sean O'Donnell,Consumer,United States,Fort Lauderdale,...,33311,South,OFF-ST-10000760,Office Supplies,Storage,Eldon Fold 'N Roll Cart System,22.3680,2,0.20,2.5164


## 16. Save Cleaned Output

In [16]:
df_clean.to_csv(clean_output_path, index=False, encoding="utf-8")

print("Clean saved:", to_relative_path(clean_output_path, PROJECT_ROOT))

Clean saved: data/clean/csv/superstore_clean.csv


## 17. Generate Ingestion Log

In [17]:
end_time = datetime.now(timezone.utc).isoformat()

ingestion_log = {
    "run_id": run_id,
    "source_name": source_name,
    "source_type": source_type,
    "input_path_or_url": to_relative_path(input_path, PROJECT_ROOT),
    "start_time": start_time,
    "end_time": end_time,
    "status": status,
    "records_read": int(records_read),
    "records_valid": int(records_valid),
    "records_invalid": int(records_invalid),
    "duplicate_rows_removed": int(duplicate_count),
    "required_fields": required_fields,
    "optional_fields": optional_fields,
    "missing_values": missing_values_all.astype(int).to_dict(),
    "required_missing_values": required_missing_values.astype(int).to_dict(),
    "optional_missing_values": optional_missing_values.astype(int).to_dict(),
    "total_missing_values": int(total_missing_values),
    "error_message": error_message,
    "raw_output_path": to_relative_path(raw_output_path, PROJECT_ROOT),
    "staging_output_path": to_relative_path(staging_output_path, PROJECT_ROOT),
    "clean_output_path": to_relative_path(clean_output_path, PROJECT_ROOT),
    "owner": owner
}

with open(log_output_path, "w", encoding="utf-8") as file:
    json.dump(ingestion_log, file, indent=4, ensure_ascii=False)

print("Log saved:", to_relative_path(log_output_path, PROJECT_ROOT))
ingestion_log

Log saved: logs/csv_ingestion_log.json


{'run_id': 'a21ea7fa-6734-477b-b3d1-e0403f4f23ce',
 'source_name': 'superstore_sales_csv',
 'source_type': 'csv',
 'input_path_or_url': 'data/sample_inputs/Superstore.csv',
 'start_time': '2026-06-14T04:51:29.519871+00:00',
 'end_time': '2026-06-14T04:51:49.769306+00:00',
 'status': 'success',
 'records_read': 9994,
 'records_valid': 9994,
 'records_invalid': 0,
 'duplicate_rows_removed': 0,
 'required_fields': ['row_id',
  'order_id',
  'order_date',
  'ship_date',
  'customer_id',
  'customer_name',
  'country',
  'city',
  'state',
  'region',
  'product_id',
  'category',
  'sub_category',
  'product_name',
  'sales',
  'quantity',
  'discount',
  'profit'],
 'optional_fields': ['ship_mode', 'segment', 'postal_code'],
 'missing_values': {'row_id': 0,
  'order_id': 0,
  'order_date': 0,
  'ship_date': 0,
  'ship_mode': 0,
  'customer_id': 0,
  'customer_name': 0,
  'segment': 0,
  'country': 0,
  'city': 0,
  'state': 0,
  'postal_code': 0,
  'region': 0,
  'product_id': 0,
  'categ

## 18. Final Output Check

In [18]:
print("Raw exists:", raw_output_path.exists())
print("Staging exists:", staging_output_path.exists())
print("Clean exists:", clean_output_path.exists())
print("Log exists:", log_output_path.exists())

print("\nOutput files:")
print(to_relative_path(raw_output_path, PROJECT_ROOT))
print(to_relative_path(staging_output_path, PROJECT_ROOT))
print(to_relative_path(clean_output_path, PROJECT_ROOT))
print(to_relative_path(log_output_path, PROJECT_ROOT))

Raw exists: True
Staging exists: True
Clean exists: True
Log exists: True

Output files:
data/raw/csv/superstore_raw.csv
data/staging/csv/superstore_staging.csv
data/clean/csv/superstore_clean.csv
logs/csv_ingestion_log.json


## 19. Summary

```text
data/raw/csv/superstore_raw.csv
data/staging/csv/superstore_staging.csv
data/clean/csv/superstore_clean.csv
logs/csv_ingestion_log.json
```